# Supplementary Tables S3 and S4 CNV segment summaries

This notebook compiles the segment-level copy-number tables used for Supplementary Tables S3 and S4 from Sequenza-derived copy-number outputs.

- Supplementary Table S3 summarizes absolute copy-number segment calls.
- Supplementary Table S4 summarizes ploidy-adjusted relative copy-number segment calls.

Relative gain/loss calls are defined relative to each sample's estimated ploidy using a threshold of 0.5 copies.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import seaborn as sns
import ast
import re
import seaborn as sns
import matplotlib.ticker as ticker
import matplotlib.patches as mpatches
import matplotlib as mpl  
from collections import OrderedDict
from pathlib import Path

pd.set_option('display.max_columns', None) 
pd.set_option('display.max_rows', 120) 

In [ ]:
new_sample_list = \
['RAP-001_65852_T',
 'RAP-001_65847_T',
 'RAP-002_14015_T',
 'RAP-002_14071_T',
 'RAP-002_14009_T',
 'RAP-002_14057_T',
 'RAP-002_14029_T',
 'RAP-003_69675_T',
 'RAP-003_69443_T',
 'RAP-003_69692_T',
 'RAP-004_70735_T',
 'RAP-004_70676_T',
 'RAP-004_70385_T',
 'RAP-004_70939_T',
 'RAP-004_70396_T',
 'RAP-004_70647_T',
 'RAP-004_70370_T',
 'RAP-004_70401_T',
 'RAP-005_179178_T',
 'RAP-005_179317_T',
 'RAP-005_179141_T',
 'RAP-005_179174_T',
 'RAP-005_179337_T',
 'RAP-005_179131_T',
 'RAP-005_179341_T',
 'RAP-006_150206_T',
 'RAP-006_150169_T',
 'RAP-006_150153_T',
 'RAP-006_150124_T',
 'RAP-006_150264_T',
 'RAP-006_150175_T',
 'RAP-006_150200_T',
 'RAP-006_150289_T',
 'RAP-006_150274_T',
 'RAP-006_150230_T',
 'RAP-006_150113_T',
 'RAP-006_150139_T',
 'RAP-006_150145_T',
 'RAP-007_245081_T',
 'RAP-007_245138_T',
 'RAP-007_244988_T',
 'RAP-007_244103_T',
 'RAP-007_245184_T',
 'RAP-007_244126_T',
 'RAP-007_245062_T',
 'RAP-007_244120_T',
 'RAP-007_245163_T',
 'RAP-007_245725_T']

In [ ]:
# paths
absolute_dir = Path("PATH_TO_ABSOLUTE_SEG_FILES")
relative_dir = Path("PATH_TO_RELATIVE_SEG_FILES")

# helpers
def parse_ids(raw_id: str):
    """
    Example:
    raw_id = 'RAP-001_65852_T'
    Patient_ID = 'RAP-001'
    Sample_ID  = 'RAP-001_65852'
    """
    patient_id = raw_id.split("_")[0]
    sample_id = raw_id.rsplit("_", 1)[0]
    return patient_id, sample_id

def chrom_to_sortable(chrom):
    """
    Convert chromosome labels to sortable values:
    1-22, X, Y, then anything else.
    """
    chrom = str(chrom).replace("chr", "")
    if chrom.isdigit():
        return (0, int(chrom))
    elif chrom == "X":
        return (1, 23)
    elif chrom == "Y":
        return (1, 24)
    else:
        return (2, chrom)

def read_seg(seg_path: Path, value_col: str, table_type: str):
    """
    Read a .seg file and return a clean dataframe.
    """
    df = pd.read_csv(seg_path, sep="\t", comment="#")

    df = df.rename(columns={
        "ID": "Raw_Sample_ID",
        "chrom": "Chromosome",
        "loc.start": "Start_Position",
        "loc.end": "End_Position",
        "seg.mean": value_col
    })

    if "Raw_Sample_ID" in df.columns and not df.empty:
        raw_id = str(df["Raw_Sample_ID"].iloc[0])
    else:
        raw_id = seg_path.stem

    patient_id, sample_id = parse_ids(raw_id)

    df["Patient_ID"] = patient_id
    df["Sample_ID"] = sample_id
    df["Segment_Length_bp"] = df["End_Position"] - df["Start_Position"] + 1
    df["Table_Type"] = table_type

    # sort helper
    df["Chromosome_Sort"] = df["Chromosome"].map(chrom_to_sortable)

    return df

# load absolute seg files
absolute_dfs = []
missing_absolute = []

for raw_sample_id in new_sample_list:
    seg_path = absolute_dir / f"{raw_sample_id}.seg"
    if seg_path.exists():
        abs_df = read_seg(
            seg_path=seg_path,
            value_col="Absolute_CN_vs_Diploid",
            table_type="Absolute"
        )

        abs_df["Absolute_Total_CN"] = abs_df["Absolute_CN_vs_Diploid"] + 2

        abs_df["Absolute_CNV_Call"] = np.select(
            [
                abs_df["Absolute_Total_CN"] == 0,
                abs_df["Absolute_Total_CN"] >= 6,
                abs_df["Absolute_CN_vs_Diploid"] < 0,
                abs_df["Absolute_CN_vs_Diploid"] > 0,
            ],
            [
                "Deep_Deletion",
                "High_Level_Amplification",
                "Loss_vs_Diploid",
                "Gain_vs_Diploid",
            ],
            default="Diploid"
        )

        absolute_dfs.append(abs_df)
    else:
        missing_absolute.append(raw_sample_id)

table_s3_df = pd.concat(absolute_dfs, ignore_index=True) if absolute_dfs else pd.DataFrame()

# load relative seg files
relative_dfs = []
missing_relative = []

for raw_sample_id in new_sample_list:
    seg_path = relative_dir / f"{raw_sample_id}.seg"
    if seg_path.exists():
        rel_df = read_seg(
            seg_path=seg_path,
            value_col="Relative_CN_vs_Ploidy",
            table_type="Relative"
        )

        # based on your Methods:
        # gain/loss relative to sample ploidy if difference >= 0.5 copies
        rel_df["Relative_CNV_Call"] = np.select(
            [
                rel_df["Relative_CN_vs_Ploidy"] <= -0.5,
                rel_df["Relative_CN_vs_Ploidy"] >= 0.5,
            ],
            [
                "Relative_Loss",
                "Relative_Gain",
            ],
            default="Neutral_vs_Ploidy"
        )

        relative_dfs.append(rel_df)
    else:
        missing_relative.append(raw_sample_id)

table_s4_df = pd.concat(relative_dfs, ignore_index=True) if relative_dfs else pd.DataFrame()

# reorder columns 
s3_cols = [
    "Patient_ID",
    "Sample_ID",
    "Raw_Sample_ID",
    "Chromosome",
    "Start_Position",
    "End_Position",
    "Segment_Length_bp",
    "Absolute_CN_vs_Diploid",
    "Absolute_Total_CN",
    "Absolute_CNV_Call",
    "Table_Type",
    "Chromosome_Sort",
]

s4_cols = [
    "Patient_ID",
    "Sample_ID",
    "Raw_Sample_ID",
    "Chromosome",
    "Start_Position",
    "End_Position",
    "Segment_Length_bp",
    "Relative_CN_vs_Ploidy",
    "Relative_CNV_Call",
    "Table_Type",
    "Chromosome_Sort",
]

table_s3_df = table_s3_df[[c for c in s3_cols if c in table_s3_df.columns]].copy()
table_s4_df = table_s4_df[[c for c in s4_cols if c in table_s4_df.columns]].copy()

# sort
if not table_s3_df.empty:
    table_s3_df = table_s3_df.sort_values(
        by=["Patient_ID", "Sample_ID", "Chromosome_Sort", "Start_Position"],
        kind="stable"
    ).drop(columns="Chromosome_Sort")

if not table_s4_df.empty:
    table_s4_df = table_s4_df.sort_values(
        by=["Patient_ID", "Sample_ID", "Chromosome_Sort", "Start_Position"],
        kind="stable"
    ).drop(columns="Chromosome_Sort")

# save as TSV
table_s3_df.to_csv("PATH_TO_OUTPUT_DIR", sep="\t", index=False)
table_s4_df.to_csv("PATH_TO_OUTPUT_DIR", sep="\t", index=False)

# report missing files
print("Missing absolute seg files:", missing_absolute)
print("Missing relative seg files:", missing_relative)
print("Table S3 shape:", table_s3_df.shape)
print("Table S4 shape:", table_s4_df.shape)

Missing absolute seg files: []
Missing relative seg files: []
Table S3 shape: (3214, 11)
Table S4 shape: (3214, 10)
